# 02 — Pré-processamento dos Dados

Dados reais são bagunçados. Aqui vamos limpar, transformar e preparar os dados para o modelo.

Etapas:
1. Tratamento de valores nulos
2. Feature Engineering (criar novas features)
3. Encoding de variáveis categóricas
4. Verificação final

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.join(os.getcwd(), '..'))
from src.preprocessor import fill_missing_values, engineer_features, encode_categoricals

df = pd.read_csv('../data/train.csv')
print(f'✅ Dados carregados: {df.shape}')

## 2. Tratamento de Valores Nulos

Aqui dividimos os nulos em dois grupos:
- **Nulos com significado**: sem garagem, sem piscina → preenchemos com 'None' ou 0
- **Nulos aleatórios**: informação simplesmente faltando → preenchemos com mediana/moda

In [ ]:
print(f'Nulos antes: {df.isnull().sum().sum()}')
df_clean = fill_missing_values(df)
print(f'Nulos depois: {df_clean.isnull().sum().sum()}')

## 3. Feature Engineering

Criamos novas features que o modelo não teria acesso diretamente, mas que fazem todo o sentido para o problema:
- **TotalSF**: área total (porão + andares)
- **HouseAge**: idade da casa
- **TotalBathrooms**: total de banheiros
- E mais!

In [ ]:
colunas_antes = df_clean.shape[1]
df_features = engineer_features(df_clean)
colunas_depois = df_features.shape[1]

print(f'Features antes: {colunas_antes}')
print(f'Features depois: {colunas_depois}')
print(f'Novas features criadas: {colunas_depois - colunas_antes}')

# Mostra as novas features
novas = ['TotalSF', 'HouseAge', 'RemodAge', 'HasGarage', 'HasPool', 'HasFireplace', 'TotalBathrooms']
df_features[novas].describe()

## 4. Encoding de Variáveis Categóricas

Modelos de ML só entendem números. Precisamos converter as colunas de texto (como 'Neighborhood', 'GarageType') em valores numéricos.

In [ ]:
categoricas_antes = df_features.select_dtypes(include='object').columns.tolist()
print(f'Colunas categóricas antes: {len(categoricas_antes)}')

df_encoded = encode_categoricals(df_features)

categoricas_depois = df_encoded.select_dtypes(include='object').columns.tolist()
print(f'Colunas categóricas depois: {len(categoricas_depois)}')
print('\n✅ Todas as colunas são agora numéricas!')
df_encoded.head()

## 5. Verificação Final

Antes de passar para a modelagem, verificamos se está tudo certo.

In [ ]:
# Separa features e alvo
y = np.log1p(df_encoded['SalePrice'])  # log-transform no alvo
X = df_encoded.drop(columns=['SalePrice', 'Id'], errors='ignore')

print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')
print(f'Nulos em X: {X.isnull().sum().sum()}')
print(f'\n✅ Dados prontos para modelagem!')